<a href="https://colab.research.google.com/github/Thanusree636/24B11CS385_DriveReady_OPC-2026/blob/main/Day04f.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q --upgrade --no-cache-dir langchain langchain-groq gspread pydantic

In [2]:
import langchain
import langchain_core
import langchain_groq
import gspread
import pydantic

print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)
print("All packages imported successfully!")

LangChain: 1.5.2
LangChain Core: 1.8.5
All packages imported successfully!


In [3]:
from langchain_groq import ChatGroq

print("ChatGroq imported successfully!")

ChatGroq imported successfully!


In [4]:
import os
from google.colab import userdata

# Load your Groq API key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Initialize the language model
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("ShopEase LLM initialized successfully!")

ShopEase LLM initialized successfully!


In [5]:
from langchain.tools import tool

# ShopEase product catalogue
PRODUCTS = {
    "wireless headphones": 1499,
    "smart watch": 2499,
    "laptop bag": 899,
    "bluetooth speaker": 1299,
    "keyboard": 699
}

# Delivery locations
DELIVERY_CITIES = [
    "hyderabad",
    "secunderabad",
    "vijayawada",
    "visakhapatnam",
    "rajahmundry",
    "kakinada"
]


@tool
def get_product_price(product_name: str) -> str:
    """Check the price and availability of a product in ShopEase.

    Use this tool when a customer asks about a product or its price.
    """
    product = product_name.lower().strip()

    if product not in PRODUCTS:
        return (
            f"{product_name} is not available. "
            f"Available products: {', '.join(PRODUCTS.keys())}"
        )

    return f"The price of {product_name} is Rs. {PRODUCTS[product]}."


@tool
def check_delivery(city: str) -> str:
    """Check whether ShopEase delivers to a particular city.

    Use this tool when a customer asks about delivery availability.
    """
    city_name = city.lower().strip()

    if city_name in DELIVERY_CITIES:
        return f"Yes, delivery is available in {city.title()}."

    return f"Sorry, delivery is currently unavailable in {city.title()}."


print("ShopEase tools created successfully!")
print("Available products:", list(PRODUCTS.keys()))

ShopEase tools created successfully!
Available products: ['wireless headphones', 'smart watch', 'laptop bag', 'bluetooth speaker', 'keyboard']


In [6]:
print(get_product_price.invoke({"product_name": "smart watch"}))

print(check_delivery.invoke({"city": "Hyderabad"}))

print(get_product_price.invoke({"product_name": "mobile phone"}))

The price of smart watch is Rs. 2499.
Yes, delivery is available in Hyderabad.
mobile phone is not available. Available products: wireless headphones, smart watch, laptop bag, bluetooth speaker, keyboard


In [7]:
from pydantic import BaseModel, Field
from typing import Literal

class ShoppingOrder(BaseModel):
    """Customer order record for ShopEase Online Store."""

    customer_name: str = Field(
        description="Full name of the customer"
    )

    product_name: Literal[
        "wireless headphones",
        "smart watch",
        "laptop bag",
        "bluetooth speaker",
        "keyboard"
    ] = Field(
        description="Product selected by the customer"
    )

    quantity: int = Field(
        description="Number of products ordered",
        ge=1
    )

    delivery_date: str = Field(
        description="Requested delivery date"
    )

    city: str = Field(
        description="Customer's delivery city"
    )

    payment_method: Literal[
        "cash on delivery",
        "upi",
        "card"
    ] = Field(
        description="Customer's payment method"
    )

print("ShoppingOrder model created successfully!")

ShoppingOrder model created successfully!


In [8]:
sample_order = ShoppingOrder(
    customer_name="Thanu",
    product_name="smart watch",
    quantity=2,
    delivery_date="30 November 2026",
    city="Hyderabad",
    payment_method="upi"
)

print(sample_order)

customer_name='Thanu' product_name='smart watch' quantity=2 delivery_date='30 November 2026' city='Hyderabad' payment_method='upi'


In [9]:
try:
    invalid_order = ShoppingOrder(
        customer_name="Ravi",
        product_name="keyboard",
        quantity=0,
        delivery_date="30 November 2026",
        city="Hyderabad",
        payment_method="upi"
    )
except Exception as e:
    print("Validation error:")
    print(e)

Validation error:
1 validation error for ShoppingOrder
quantity
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/greater_than_equal


In [10]:
structured_llm = llm.with_structured_output(ShoppingOrder)

print("Structured output model is ready!")

Structured output model is ready!


In [11]:
customer_messages = [
    "Hey, I'm Ravi. Need 2 smart watches delivered to Hyderabad on 30 November. I'll pay using UPI.",

    "Can u send me one laptop bag to Vijayawada by 2nd December? Name is Priya, cash on delivery.",

    "This is Arjun. I want three bluetooth speakers. Deliver to Visakhapatnam on 5 December. Payment by card.",

    "Hi, I'm Sneha, need a keyboard. Just one piece, send it to Kakinada by 3 December. I'll use UPI.",

    "Please book 2 wireless headphones for Rahul, delivery to Hyderabad on 10 December, payment COD."
]

for i, message in enumerate(customer_messages, 1):
    print(f"\nTEST {i}")
    print("Customer:", message)

    try:
        result = structured_llm.invoke(message)
        print("Extracted order:", result)
    except Exception as e:
        print("Extraction error:", e)


TEST 1
Customer: Hey, I'm Ravi. Need 2 smart watches delivered to Hyderabad on 30 November. I'll pay using UPI.
Extracted order: customer_name='Ravi' product_name='smart watch' quantity=2 delivery_date='30 November' city='Hyderabad' payment_method='upi'

TEST 2
Customer: Can u send me one laptop bag to Vijayawada by 2nd December? Name is Priya, cash on delivery.
Extracted order: customer_name='Priya' product_name='laptop bag' quantity=1 delivery_date='2nd December' city='Vijayawada' payment_method='cash on delivery'

TEST 3
Customer: This is Arjun. I want three bluetooth speakers. Deliver to Visakhapatnam on 5 December. Payment by card.
Extracted order: customer_name='Arjun' product_name='bluetooth speaker' quantity=3 delivery_date='5 December' city='Visakhapatnam' payment_method='card'

TEST 4
Customer: Hi, I'm Sneha, need a keyboard. Just one piece, send it to Kakinada by 3 December. I'll use UPI.
Extracted order: customer_name='Sneha' product_name='keyboard' quantity=1 delivery_dat

In [12]:
from google.colab import auth
import gspread
from google.auth import default

# Authenticate Google account
auth.authenticate_user()

# Connect to Google Sheets
creds, _ = default()
gc = gspread.authorize(creds)

# Create a new spreadsheet
sh = gc.create("ShopEase Customer Orders")

# Select the first worksheet
ws = sh.sheet1

# Add column headings
ws.append_row([
    "customer_name",
    "product_name",
    "quantity",
    "delivery_date",
    "city",
    "payment_method",
    "total_price"
])

print("Google Sheet created successfully!")
print("Sheet URL:", sh.url)

Google Sheet created successfully!
Sheet URL: https://docs.google.com/spreadsheets/d/1saBuT7xydl5EjI0ssgWAkthOh8Z6XkFMx_wrvDaLwaA


In [13]:

from langchain.tools import tool

@tool
def log_order(
    customer_name: str,
    product_name: str,
    quantity: int,
    delivery_date: str,
    city: str,
    payment_method: str
) -> str:
    """Save a valid customer order to the ShopEase Google Sheet.

    Use this tool only after collecting all required customer details
    and receiving the customer's confirmation.

    Args:
        customer_name: Full name of the customer.
        product_name: Product selected from the catalogue.
        quantity: Number of products, at least 1.
        delivery_date: Requested delivery date.
        city: Customer's delivery city.
        payment_method: UPI, card, or cash on delivery.
    """

    try:
        # Validate the order
        order = ShoppingOrder(
            customer_name=customer_name.strip(),
            product_name=product_name.lower().strip(),
            quantity=quantity,
            delivery_date=delivery_date.strip(),
            city=city.strip(),
            payment_method=payment_method.lower().strip()
        )

        if not order.customer_name:
            return "Customer name is required."

        if not order.delivery_date:
            return "Delivery date is required."

        if order.city.lower() not in DELIVERY_CITIES:
            return f"Delivery is unavailable in {order.city}."

        # Calculate total
        total = PRODUCTS[order.product_name] * order.quantity

        # Save order to Google Sheets
        ws.append_row([
            order.customer_name,
            order.product_name,
            order.quantity,
            order.delivery_date,
            order.city,
            order.payment_method,
            total
        ])

        return (
            f"Order saved successfully! "
            f"{order.quantity} x {order.product_name}, "
            f"Total: Rs. {total}."
        )

    except Exception as e:
        return f"Order could not be saved: {e}"


print("ShopEase log_order tool created successfully!")


ShopEase log_order tool created successfully!


In [14]:
result = log_order.invoke({
    "customer_name": "Test Customer",
    "product_name": "smart watch",
    "quantity": 2,
    "delivery_date": "30 November 2026",
    "city": "Hyderabad",
    "payment_method": "upi"
})

print(result)

Order saved successfully! 2 x smart watch, Total: Rs. 4998.


In [15]:

from langchain.agents import create_agent

SYSTEM_PROMPT = """
ROLE:
You are ShopEase Assistant, a friendly online shopping assistant.

STORE INFORMATION:
You work for ShopEase Online Store.

AVAILABLE PRODUCTS:
- Wireless Headphones
- Smart Watch
- Laptop Bag
- Bluetooth Speaker
- Keyboard

RULES:
1. Always use get_product_price to check product prices.
2. Always use check_delivery to verify delivery locations.
3. Never invent product prices or availability.
4. Collect these details before placing an order:
   - Customer name
   - Product name
   - Quantity
   - Delivery date
   - City
   - Payment method
5. Quantity must be at least 1.
6. Check that the product exists in the catalogue.
7. If any information is missing, ask the customer.
8. Before saving an order, show the order summary and ask
   the customer for confirmation.
9. Only call log_order after the customer explicitly confirms.
10. Never claim an order was saved unless log_order succeeds.
11. Be friendly, clear and concise.
"""

agent = create_agent(
    model=llm,
    tools=[
        get_product_price,
        check_delivery,
        log_order
    ],
    system_prompt=SYSTEM_PROMPT
)

print("ShopEase AI Agent created successfully!")


ShopEase AI Agent created successfully!


In [16]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of a smart watch?"
        }
    ]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the price of a smart watch?
================================== Ai Message ==================================
Tool Calls:
  get_product_price (fc_273fef9d-532e-4839-8f5e-e1cf4ed7dc0c)
 Call ID: fc_273fef9d-532e-4839-8f5e-e1cf4ed7dc0c
  Args:
    product_name: Smart Watch
================================= Tool Message =================================
Name: get_product_price

The price of Smart Watch is Rs. 2499.
================================== Ai Message ==================================

The Smart Watch is priced at **₹2,499**. Let me know if you’d like to add it to your cart or need any other information!


In [17]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """
ROLE:
You are ShopEase Assistant, a friendly online shopping assistant.

STORE INFORMATION:
You work for ShopEase Online Store.

AVAILABLE PRODUCTS:
- Wireless Headphones
- Smart Watch
- Laptop Bag
- Bluetooth Speaker
- Keyboard

RULES:
1. Always use get_product_price to check product prices.
2. Always use check_delivery to verify delivery locations.
3. Never invent product prices or availability.
4. Collect these details before placing an order:
   - Customer name
   - Product name
   - Quantity
   - Delivery date
   - City
   - Payment method
5. Quantity must be at least 1.
6. Check that the product exists in the catalogue.
7. If any information is missing, ask the customer.
8. Before saving an order, show the order summary and ask
   the customer for confirmation.
9. Only call log_order after the customer explicitly confirms.
10. Never claim an order was saved unless log_order succeeds.
11. Be friendly, clear and concise.
"""

agent = create_agent(
    model=llm,
    tools=[get_product_price, check_delivery, log_order],
    system_prompt=SYSTEM_PROMPT
)

print("ShopEase AI Agent created successfully!")

ShopEase AI Agent created successfully!


In [18]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of a smart watch?"
        }
    ]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the price of a smart watch?
================================== Ai Message ==================================
Tool Calls:
  get_product_price (fc_342c4d99-8a5e-429b-85b8-124edfdabf36)
 Call ID: fc_342c4d99-8a5e-429b-85b8-124edfdabf36
  Args:
    product_name: Smart Watch
================================= Tool Message =================================
Name: get_product_price

The price of Smart Watch is Rs. 2499.
================================== Ai Message ==================================

The Smart Watch is priced at **Rs. 2499**. Let me know if you’d like to check delivery availability or place an order!


In [19]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "I want to order 2 smart watches."
        }
    ]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

I want to order 2 smart watches.
================================== Ai Message ==================================

Sure thing! Before I place the order, could you please provide the following details?

1. **Your full name**  
2. **Delivery city**  
3. **Preferred delivery date** (e.g., 2026‑12‑05)  
4. **Payment method** (UPI, card, or cash on delivery)

Once I have that, I’ll check the price and confirm delivery availability.


In [20]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            My name is Ravi Kumar.
            I want 2 smart watches.
            Deliver them to Hyderabad on 5 December 2026.
            My payment method is UPI.
            """
        }
    ]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================


            My name is Ravi Kumar.
            I want 2 smart watches.
            Deliver them to Hyderabad on 5 December 2026.
            My payment method is UPI.
            
================================== Ai Message ==================================
Tool Calls:
  get_product_price (fc_65059e4e-1997-48bb-857a-f5ece6dc2664)
 Call ID: fc_65059e4e-1997-48bb-857a-f5ece6dc2664
  Args:
    product_name: Smart Watch
================================= Tool Message =================================
Name: get_product_price

The price of Smart Watch is Rs. 2499.
================================== Ai Message ==================================
Tool Calls:
  check_delivery (fc_7c3a5d70-6314-4aea-97d2-0f8c72860a0b)
 Call ID: fc_7c3a5d70-6314-4aea-97d2-0f8c72860a0b
  Args:
    city: Hyderabad
================================= Tool Message =================================
Name: check_delivery

Yes, delivery is 

In [21]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            My name is Ravi Kumar.
            I want 2 smart watches.
            Deliver them to Hyderabad on 5 December 2026.
            My payment method is UPI.
            """
        },
        {
            "role": "assistant",
            "content": """
            Here is your order summary:
            Ravi Kumar, 2 smart watches, Hyderabad,
            5 December 2026, UPI, total Rs. 4998.
            Is everything correct?
            """
        },
        {
            "role": "user",
            "content": "Yes, I confirm. Please place my order."
        }
    ]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================


            My name is Ravi Kumar.
            I want 2 smart watches.
            Deliver them to Hyderabad on 5 December 2026.
            My payment method is UPI.
            
================================== Ai Message ==================================


            Here is your order summary:
            Ravi Kumar, 2 smart watches, Hyderabad,
            5 December 2026, UPI, total Rs. 4998.
            Is everything correct?
================================ Human Message =================================

Yes, I confirm. Please place my order.
================================== Ai Message ==================================
Tool Calls:
  get_product_price (fc_485eec99-d98d-4559-b338-471db626b514)
 Call ID: fc_485eec99-d98d-4559-b338-471db626b514
  Args:
    product_name: Smart Watch
================================= Tool Message =================================
Name: get_product_price

The pri

In [22]:
records = ws.get_all_records()

for record in records:
    print(record)

{'customer_name': 'Test Customer', 'product_name': 'smart watch', 'quantity': 2, 'delivery_date': '30 November 2026', 'city': 'Hyderabad', 'payment_method': 'upi', 'total_price': 4998}
{'customer_name': 'Ravi Kumar', 'product_name': 'smart watch', 'quantity': 2, 'delivery_date': '5 December 2026', 'city': 'Hyderabad', 'payment_method': 'upi', 'total_price': 4998}


In [23]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            My name is Anjali.
            I want 0 smart watches.
            Deliver to Hyderabad on 10 December 2026.
            Payment method is UPI.
            """
        }
    ]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================


            My name is Anjali.
            I want 0 smart watches.
            Deliver to Hyderabad on 10 December 2026.
            Payment method is UPI.
            
================================== Ai Message ==================================
Tool Calls:
  check_delivery (fc_60df3633-4ffa-4040-bbdc-e40d82289386)
 Call ID: fc_60df3633-4ffa-4040-bbdc-e40d82289386
  Args:
    city: Hyderabad
================================= Tool Message =================================
Name: check_delivery

Yes, delivery is available in Hyderabad.
================================== Ai Message ==================================

Hi Anjali! 👋  
I see you’d like to order **Smart Watch**.  
Could you let me know how many you’d like? (The quantity must be at least 1.)


In [24]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "I want to buy an iPhone 17. What is its price?"
        }
    ]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

I want to buy an iPhone 17. What is its price?
================================== Ai Message ==================================

I’m sorry, but **iPhone 17** isn’t part of our current catalogue.  
We do have a range of other great gadgets—like Wireless Headphones, a Smart Watch, a Laptop Bag, a Bluetooth Speaker, or a Keyboard.  

Would you like to hear the price of any of those, or perhaps explore a different product?


In [25]:
from langchain.tools import tool

@tool
def tool1(product_name: str) -> str:
    """does stuff"""
    product = product_name.lower().strip()

    if product not in PRODUCTS:
        return f"{product_name} is not available."

    return f"The price of {product_name} is Rs. {PRODUCTS[product]}."

print("Experimental tool created!")

Experimental tool created!


In [26]:
experimental_agent = create_agent(
    model=llm,
    tools=[tool1, check_delivery, log_order],
    system_prompt="""
    You are a helpful ShopEase shopping assistant.
    Help customers check product prices and place orders.
    Use the available tools whenever necessary.
    """
)

print("Experimental agent created successfully!")

Experimental agent created successfully!


In [27]:
response = experimental_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of a smart watch?"
        }
    ]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the price of a smart watch?
================================== Ai Message ==================================
Tool Calls:
  tool1 (fc_ad9abc25-d019-4593-a1a0-029050ac4fcc)
 Call ID: fc_ad9abc25-d019-4593-a1a0-029050ac4fcc
  Args:
    product_name: smart watch
================================= Tool Message =================================
Name: tool1

The price of smart watch is Rs. 2499.
================================== Ai Message ==================================

The price of a smart watch is **Rs. 2499**. If you’d like to place an order or need more details, just let me know!


In [28]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of a smart watch and is delivery available in Hyderabad?"
        }
    ]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the price of a smart watch and is delivery available in Hyderabad?
================================== Ai Message ==================================
Tool Calls:
  get_product_price (fc_179b846f-af96-4998-a900-bbece24b0a8d)
 Call ID: fc_179b846f-af96-4998-a900-bbece24b0a8d
  Args:
    product_name: Smart Watch
================================= Tool Message =================================
Name: get_product_price

The price of Smart Watch is Rs. 2499.
================================== Ai Message ==================================
Tool Calls:
  check_delivery (fc_95947c4d-c0b9-4fb5-856c-41626c67a27c)
 Call ID: fc_95947c4d-c0b9-4fb5-856c-41626c67a27c
  Args:
    city: Hyderabad
================================= Tool Message =================================
Name: check_delivery

Yes, delivery is available in Hyderabad.
================================== Ai Message ==================================



In [29]:
messages = response["messages"]

tool_calls = sum(
    len(getattr(message, "tool_calls", []) or [])
    for message in messages
)

model_calls = sum(
    1 for message in messages
    if message.__class__.__name__ == "AIMessage"
)

print("Total AI model calls:", model_calls)
print("Total tool calls:", tool_calls)

Total AI model calls: 3
Total tool calls: 2


In [30]:
agent = create_agent(
    model=llm,
    tools=[get_product_price, check_delivery, log_order],
    system_prompt=SYSTEM_PROMPT
)

print("Original ShopEase agent restored successfully!")

Original ShopEase agent restored successfully!
